# Thesis-decision ladder for Garrido-Rios MFSC

This notebook is the paper-facing runner for the thesis-decision ladder. It does not make DMLPA the protagonist. It runs the versioned repo scripts so Colab/Kaggle results match the committed code.

Contract discipline:

- `thesis_factorized = MultiDiscrete([6, 3])` is the strict thesis-decision RL action surface: common `I_{t,S}` level plus `S`.
- `factorized = MultiDiscrete([6, 6, 6, 3])` is only the declared per-node extension.
- Static ladder results must be compared within the same action space before claiming adaptive RL value.


In [ ]:
# =====================
# 0) Run config
# =====================

RUN_PROFILE = "debug"  # "debug" for smoke; "serious" for paper-grade runs.

GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"
REPO_DIR_COLAB = "/content/scresia"
REPO_DIR_KAGGLE = "/kaggle/working/scresia"

LOCAL_OUTPUT_DIR = "/content/scresia_thesis_ladder_outputs"
KAGGLE_OUTPUT_DIR = "/kaggle/working/scresia_thesis_ladder_outputs"

RUNTIME_PIP_PACKAGES = [
    "simpy>=4.1",
    "numpy>=1.26",
    "gymnasium>=0.29",
    "stable-baselines3>=2.3",
    "sb3-contrib>=2.3",
    "pandas>=2.2",
    "matplotlib>=3.8",
]

REWARD_MODE = "ReT_cd_v1"
RISK_LEVEL = "increased"
OBSERVATION_VERSION = "v5"
OBSERVATION_MODE = "env_sdm_history_reward"
STEP_SIZE_HOURS = 168.0
STOCHASTIC_PT = True

if RUN_PROFILE == "debug":
    L0_L1A_REPLICATIONS = 1
    L1B_SCREENING_REPLICATIONS = 1
    L1B_TOP_K = 3
    L1B_TOP_REPLICATIONS = 1
    MAX_STEPS = 4
    PPO_TIMESTEPS = 16
    PPO_EVAL_EPISODES = 1
else:
    L0_L1A_REPLICATIONS = 30
    L1B_SCREENING_REPLICATIONS = 5
    L1B_TOP_K = 20
    L1B_TOP_REPLICATIONS = 30
    MAX_STEPS = 260
    PPO_TIMESTEPS = 200_000
    PPO_EVAL_EPISODES = 20

RUN_L0_L1A = True
RUN_L1B_PER_NODE = False  # Turn on for the overnight per-node extension.
RUN_PPO_THESIS_FACTORIZED = False  # Turn on after static L0/L1a results exist.
RUN_PPO_PER_NODE = False  # Turn on only if L1b shows value.

print({
    "run_profile": RUN_PROFILE,
    "l0_l1a_replications": L0_L1A_REPLICATIONS,
    "l1b_screening_replications": L1B_SCREENING_REPLICATIONS,
    "l1b_top_k": L1B_TOP_K,
    "max_steps": MAX_STEPS,
    "ppo_timesteps": PPO_TIMESTEPS,
})


In [ ]:
# =======================================
# 1) Portable Colab/Kaggle repo setup
# =======================================

import json
import os
import random
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = Path("/kaggle/working").exists()
if not (IN_COLAB or IN_KAGGLE):
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {cmd}")
    return result

if IN_COLAB or IN_KAGGLE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PIP_PACKAGES])
else:
    print("local run: skipping pip install")

if IN_KAGGLE:
    REPO_DIR = Path(REPO_DIR_KAGGLE)
    output_root = Path(KAGGLE_OUTPUT_DIR)
elif IN_COLAB:
    REPO_DIR = Path(REPO_DIR_COLAB)
    output_root = Path(LOCAL_OUTPUT_DIR)
else:
    REPO_DIR = Path.cwd()
    output_root = Path.cwd() / "outputs" / "thesis_decision_ladder_colab"

if IN_COLAB or IN_KAGGLE:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(REPO_DIR)])
    sys.path.insert(0, str(REPO_DIR.parent))
    sys.path.insert(0, str(REPO_DIR))
else:
    sys.path.insert(0, str(REPO_DIR.parent))
    sys.path.insert(0, str(REPO_DIR))

output_root.mkdir(parents=True, exist_ok=True)
print("repo:", REPO_DIR)
print("outputs:", output_root)
print("git hash:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())


In [ ]:
# ==========================
# 2) Run static ladder
# ==========================

ladder_output_root = output_root / "ladder"
ladder_output_root.mkdir(parents=True, exist_ok=True)

common_args = [
    sys.executable,
    "scripts/run_thesis_decision_ladder.py",
    "--output-root", str(ladder_output_root),
    "--reward-mode", REWARD_MODE,
    "--risk-level", RISK_LEVEL,
    "--observation-version", OBSERVATION_VERSION,
    "--observation-mode", OBSERVATION_MODE,
    "--step-size-hours", str(STEP_SIZE_HOURS),
    "--max-steps", str(MAX_STEPS),
]
if STOCHASTIC_PT:
    common_args.append("--stochastic-pt")

if RUN_L0_L1A:
    label = f"{RUN_PROFILE}_L0_L1a_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    run_cmd(common_args + [
        "--label", label,
        "--levels", "L0_garrido", "L1a_uniform_IxS",
        "--replications", str(L0_L1A_REPLICATIONS),
    ], cwd=REPO_DIR)
    l0_l1a_dir = ladder_output_root / label
    print("L0/L1a output:", l0_l1a_dir)
else:
    l0_l1a_dir = None

if RUN_L1B_PER_NODE:
    label = f"{RUN_PROFILE}_L1b_per_node_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    run_cmd(common_args + [
        "--label", label,
        "--levels", "L1b_per_node_IxS",
        "--replications", str(L0_L1A_REPLICATIONS),
        "--l1b-screening-replications", str(L1B_SCREENING_REPLICATIONS),
        "--l1b-top-k", str(L1B_TOP_K),
        "--l1b-top-replications", str(L1B_TOP_REPLICATIONS),
    ], cwd=REPO_DIR)
    l1b_dir = ladder_output_root / label
    print("L1b output:", l1b_dir)
else:
    l1b_dir = None


In [ ]:
# ==========================================
# 3) Load and inspect static ladder summaries
# ==========================================

import pandas as pd

summary_dirs = [p for p in [l0_l1a_dir, l1b_dir] if p is not None]
for run_dir in summary_dirs:
    print("\n===", run_dir.name, "===")
    summary = json.loads((run_dir / "summary.json").read_text())
    print("episode_count:", summary["episode_count"])
    print("policy_count:", summary["policy_count"])
    print("best_policy_by_fill_rate:")
    print(json.dumps(summary["best_policy_by_fill_rate"], indent=2))
    policy_df = pd.read_csv(run_dir / "policy_summary.csv")
    display(policy_df.sort_values(["fill_rate_order_level_mean", "order_level_ret_mean"], ascending=False).head(20))


In [ ]:
# ================================================
# 4) Optional PPO adaptive runs on declared contracts
# ================================================

ppo_output_root = output_root / "ppo"
ppo_output_root.mkdir(parents=True, exist_ok=True)

ppo_common = [
    sys.executable,
    "scripts/run_thesis_decision_ppo_smoke.py",
    "--output-root", str(ppo_output_root),
    "--reward-mode", REWARD_MODE,
    "--risk-level", RISK_LEVEL,
    "--observation-version", OBSERVATION_VERSION,
    "--observation-mode", OBSERVATION_MODE,
    "--step-size-hours", str(STEP_SIZE_HOURS),
    "--max-steps", str(MAX_STEPS),
    "--train-timesteps", str(PPO_TIMESTEPS),
    "--eval-episodes", str(PPO_EVAL_EPISODES),
    "--n-steps", "1024" if RUN_PROFILE == "serious" else "64",
    "--batch-size", "256" if RUN_PROFILE == "serious" else "32",
    "--n-epochs", "10" if RUN_PROFILE == "serious" else "2",
]
if STOCHASTIC_PT:
    ppo_common.append("--stochastic-pt")

if RUN_PPO_THESIS_FACTORIZED:
    label = f"{RUN_PROFILE}_ppo_thesis_factorized_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    run_cmd(ppo_common + [
        "--label", label,
        "--action-space-mode", "thesis_factorized",
        "--inventory-period-mode", "thesis_strict",
    ], cwd=REPO_DIR)
    print("PPO thesis_factorized output:", ppo_output_root / label)

if RUN_PPO_PER_NODE:
    label = f"{RUN_PROFILE}_ppo_per_node_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    run_cmd(ppo_common + [
        "--label", label,
        "--action-space-mode", "factorized",
        "--inventory-period-mode", "per_node",
    ], cwd=REPO_DIR)
    print("PPO per-node output:", ppo_output_root / label)
